In [1]:
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from loglizer import dataloader, preprocessing
import numpy as np
import pandas as pd

struct_log = '/root/success/log/HDFS/HDFS_100k.log_structured.csv'  # 结构化日志文件
label_file = '/root/success/log/HDFS/anomaly_label.csv'  # 异常标签文件

if __name__ == '__main__':
    # 加载有标签的数据
    (x_train, y_train), (x_test, y_test) = dataloader.load_HDFS(struct_log,
                                                                label_file=label_file,
                                                                window='session',
                                                                train_ratio=0.5,
                                                                split_type='uniform')

    feature_extractor = preprocessing.FeatureExtractor()
    x_train = feature_extractor.fit_transform(x_train, term_weighting='tf-idf')
    x_test = feature_extractor.transform(x_test)

    # 初始化XGBoost分类器
    model = XGBClassifier(random_state=42)

    # 使用交叉验证评估模型
    scores = cross_val_score(model, x_train, y_train, cv=5, scoring='accuracy')

    # 输出交叉验证结果
    print("Cross Validation Scores:", scores)
    print("Mean Accuracy:", np.mean(scores))

    # 训练模型
    model.fit(x_train, y_train)

    # 模型评估
    y_pred = model.predict(x_test)
    print("Final Evaluation")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(y_test, y_pred))


====== Input data summary ======
Loading /root/success/log/HDFS/HDFS_100k.log_structured.csv
156 157
Total: 7940 instances, 313 anomaly, 7627 normal
Train: 3969 instances, 156 anomaly, 3813 normal
Test: 3971 instances, 157 anomaly, 3814 normal

====== Transformed train data summary ======
Train data shape: 3969-by-14

====== Transformed test data summary ======
Test data shape: 3971-by-14

Cross Validation Scores: [0.97355164 0.97607053 0.97607053 0.97355164 0.97477932]
Mean Accuracy: 0.9748047303070633
Final Evaluation
Accuracy: 0.9773356837068748
Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3814
           1       1.00      0.43      0.60       157

    accuracy                           0.98      3971
   macro avg       0.99      0.71      0.79      3971
weighted avg       0.98      0.98      0.97      3971

